In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
import math as math
import os

In [ ]:
# Load the data
directory='results/synth10/pan_zoom_scenario/'


# Define the custom color palette 
# colors = ['black', 'red', 'green', 'blue'] 
# colors = ['black', '#253494', '#2c7fb8', '#41b6c4', '#a1dab4', 'red'] 
colors = ['black', '#2ca02c', '#ff7f0e', '#0000FF', '#d62728', '#00008B']
colors_competitors= ['#0000FF', '#d62728', '#2ca02c']
fontsize=18

selected_error_bounds =[0, 0.01, 0.02, 0.05, 0.1, 0.2]
selected_measure_cols = [4]

default_measure_cols = 4
default_error_bound = 0.01

In [ ]:
# Read all CSV files from the selected directory
def load_experimental_data(directory):
    all_data = []
    
    # Create a Path object for the directory
    dir_path = Path(directory)
    
    # Iterate through all CSV files in the directory
    for file_path in dir_path.glob('*.csv'):
        stem = file_path.stem
        mcols = None
        error_bound = None
        run = None
        try:
            parts = stem.split('_')
            for part in parts:
                if part.startswith('mcols'):
                    mcols = int(part.replace('mcols', ''))
                elif part.startswith('error'):
                    error_bound = float(part.replace('error', ''))
                elif part.startswith('run'):
                    run = int(part.replace('run', ''))
        except Exception as e:
            print(f"Warning: Could not parse filename {stem}: {e}")
            continue
        # Filter by selected_measure_cols
        if mcols not in selected_measure_cols:
            continue
        if error_bound not in selected_error_bounds:
            continue
        df = pd.read_csv(file_path)
        df['measure_cols'] = mcols
        df['errorBound'] = error_bound
        df['run'] = run
        all_data.append(df)
    
    # Combine all dataframes
    return pd.concat(all_data, ignore_index=True)


In [ ]:
# Function to plot the aggregated results
def plot_aggregated_error_per_column(directory, y_col, measure_cols, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Filter by measure_cols, then group by errorBound and query number, then calculate mean and std
    agg_df = df[df['measure_cols'] == measure_cols].groupby(['errorBound', 'i']).agg(
        mean_time=(y_col, 'mean'),
        std_time=(y_col, 'std'),
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]

    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create query index for x-axis (assuming 'i' column represents query sequence)
    plt.figure(figsize=(12, 5))

    # Plot a line for each error bound without markers
    for error_bound in agg_df['errorBound'].unique():
        data = agg_df[agg_df['errorBound'] == error_bound]
        label = 'Exact' if error_bound == 0 else f'{error_bound * 100:.0f}%'
        plt.plot(data['i'], data['mean_time'], 
                 label=label, 
                 linewidth=2,
                 color=color_palette[error_bound])

    if not include_initialization:
        plt.xlim((1,99))
    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel(y_col, fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize = fontsize, frameon=False)  # Remove the box around the legend
    
    # Remove the grid lines behind the plot
    plt.grid(False)

    # Rotate x-axis labels if there are many queries
    plt.xticks(fontsize = fontsize, rotation=0)
    plt.yticks(fontsize = fontsize)

    # Adjust x-axis to start from 1 if initialization is excluded
    if not include_initialization:
        plt.xlim(left=1)

    plt.tight_layout()
    plt.show()


# Function to plot the line chart along with a bar chart showing the summation of time compared to all errors
def plot_aggregated_error_per_column_with_bars(directory, y_col, measure_cols, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Group by errorBound and query number, then calculate mean and std
    agg_df = df[df['measure_cols'] == measure_cols].groupby(['errorBound', 'i']).agg(
        mean_time=(y_col, 'mean'),
        std_time=(y_col, 'std')
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]

    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create a figure with two subplots side by side with different sizes
    fig = plt.figure(figsize=(15, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[3, 1])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # Plot a line for each error bound without markers on the first subplot
    for error_bound in agg_df['errorBound'].unique():
        data = agg_df[agg_df['errorBound'] == error_bound]
        label = 'Exact' if error_bound == 0 else f'{error_bound * 100:.0f}%'
        ax1.plot(data['i'], data['mean_time'], 
                 label=label, 
                 linewidth=2,
                 color=color_palette[error_bound])

    ax1.set_xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    ax1.set_ylabel(y_col, fontsize=fontsize, fontweight='bold')
    ax1.legend(fontsize = fontsize, frameon=False)  # Remove the box around the legend
    
    # Remove the grid lines behind the plot
    ax1.grid(False)

    # Rotate x-axis labels if there are many queries
    ax1.tick_params(axis='x', labelsize=fontsize, rotation=0)
    ax1.tick_params(axis='y', labelsize=fontsize)

    # Adjust x-axis to start from 1 if initialization is excluded
    if not include_initialization:
        ax1.set_xlim((1,99))

    # Create the bar chart on the second subplot
    total_times = agg_df.groupby('errorBound')['mean_time'].sum().reset_index()
    total_times['label'] = total_times['errorBound'].apply(lambda x: 'Exact' if x == 0 else f'{x * 100:.0f}%')
    bars = ax2.bar(total_times['label'], total_times['mean_time'], alpha=0.7)

    # Set the colors of the bars to match the line colors
    for bar, error_bound in zip(bars, total_times['errorBound']):
        bar.set_color(color_palette[error_bound])

    ax2.set_xlabel('Error Bound', fontsize=fontsize, fontweight='bold')
    ax2.set_ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')
    ax2.tick_params(axis='x', labelsize=fontsize, rotation=0)
    ax2.tick_params(axis='y', labelsize=fontsize)

    plt.tight_layout()
    plt.show()

# Function to plot only the bar chart showing the summation of time compared to all errors
def plot_aggregated_error_per_column_bars_only(directory, y_col, measure_cols, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Group by errorBound and query number, then calculate mean and std
    agg_df = df[df['measure_cols'] == measure_cols].groupby(['errorBound', 'i']).agg(
        mean_time=(y_col, 'mean'),
        std_time=(y_col, 'std')
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]

    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create query index for x-axis (assuming 'i' column represents query sequence)
    plt.figure(figsize=(12, 5))
    
    # Create the bar chart
    total_times = agg_df.groupby('errorBound')['mean_time'].sum().reset_index()
    total_times['label'] = total_times['errorBound'].apply(lambda x: 'Exact' if x == 0 else f'{x * 100:.0f}%')
    bars = plt.bar(total_times['label'], total_times['mean_time'], alpha=0.7)

    # Set the colors of the bars to match the line colors
    for bar, error_bound in zip(bars, total_times['errorBound']):
        bar.set_color(color_palette[error_bound])

    plt.xlabel('Error Bound', fontsize=fontsize, fontweight='bold')
    plt.ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')

    # Rotate x-axis labels if there are many queries
    plt.xticks(fontsize = fontsize, rotation=0)
    plt.yticks(fontsize = fontsize)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_aggregated_error_per_column(directory, 'Time (sec)', default_measure_cols)
plot_aggregated_error_per_column_with_bars(directory, 'Time (sec)', default_measure_cols, include_initialization=False)
plot_aggregated_error_per_column_bars_only(directory, 'Time (sec)', default_measure_cols, include_initialization=False)

In [ ]:
plot_aggregated_error_per_column(directory, 'I/Os', default_measure_cols)
plot_aggregated_error_per_column_with_bars(directory, 'I/Os', default_measure_cols, include_initialization=False)
plot_aggregated_error_per_column_bars_only(directory, 'I/Os', default_measure_cols, include_initialization=False)

In [ ]:

def calculate_relative_errors(df_approx, df_exact):
    df_approx.reset_index(inplace=True)
    df_approx['relative_error'] = (abs(df_approx["conf_ub"] - df_approx["conf_lb"]) / 2) / df_exact["ground_truth"]
    return df_approx


# Function to plot the aggregated results
def plot_relative_error_per_error_bound(directory, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        conf_lb=('Confidence Interval LB', 'mean'),
        conf_ub=('Confidence Interval UB', 'mean'),
        ground_truth=("Query Result Sum", 'mean')
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]

    df_exact =  agg_df[agg_df['errorBound'] == 0]
    df_exact.reset_index(inplace=True)
    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create query index for x-axis (assuming 'i' column represents query sequence)
    plt.figure(figsize=(12, 5))

    # Plot a line for each error bound without markers
    for error_bound in agg_df['errorBound'].unique():
        if(error_bound == 0): continue;
        data = agg_df[agg_df['errorBound'] == error_bound]
        calculate_relative_errors(data, df_exact)
        label = 'Exact' if error_bound == 0 else f'{error_bound * 100:.0f}%'
        plt.plot(data['i'], data['relative_error'], 
                 label=label, 
                 linewidth=2,
                 color=color_palette[error_bound])

    if not include_initialization:
        plt.xlim((1,99))
    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel('Relative Error', fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize = fontsize, frameon=False)  # Remove the box around the legend
    
    # Remove the grid lines behind the plot
    plt.grid(False)

    # Rotate x-axis labels if there are many queries
    plt.xticks(fontsize = fontsize, rotation=0)
    plt.yticks(fontsize = fontsize)

    # Adjust x-axis to start from 1 if initialization is excluded
    if not include_initialization:
        plt.xlim(left=1)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_relative_error_per_error_bound(directory, include_initialization=False)

In [ ]:
def compare_exact_vs_approx_results(directory, include_initialization=False, approx_error_bound=0.05):
    # Load data using existing function 
    df = load_experimental_data(directory)
    df = df[df['run'] == 1]
     # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        conf_lb=('Confidence Interval LB', 'mean'),
        conf_ub=('Confidence Interval UB', 'mean'),
        error_bound=('Error Bound', 'mean'),
        val=('Query Result Sum', 'mean'),
    ).reset_index()
    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]
        
    # Get exact and approximate results
    exact_df = agg_df[agg_df['errorBound'] == 0.0]
    approx_df = agg_df[agg_df['errorBound'] == approx_error_bound]
    
    # Initialize results storage
    results = []
    
    # For each query in the approximate results
    for i in range(len(approx_df)):
        approx_row = approx_df.iloc[i]
        
        # Find matching query in exact results
        exact_row = exact_df[exact_df['i'] == approx_row['i']]
        
        if len(exact_row) == 0:
            continue
            
        exact_sum = exact_row.iloc[0]['val']
        
        # Get confidence interval and error bound
        ci_lb = approx_row['conf_lb']
        ci_ub = approx_row['conf_ub']
        error_bound = approx_row['error_bound']
        
        # Check if exact sum falls within confidence interval
        within_ci = ci_lb <= exact_sum <= ci_ub
        if(not within_ci):
            print(f"Query {i} - Exact: {exact_sum}, CI: ({ci_lb}, {ci_ub})")
        # Calculate actual relative error 
        if exact_sum != 0:
            actual_error = abs(((ci_lb + ci_ub)/2 - exact_sum) / exact_sum)
        else:
            actual_error = float('inf')
            
        results.append({
            'query_index': i,
            'exact_sum': exact_sum,
            'estimated_sum': (ci_lb + ci_ub)/2,
            'ci_lb': ci_lb,
            'ci_ub': ci_ub, 
            'error_bound': error_bound,
            'actual_error': actual_error,
            'within_ci': within_ci
        })
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Plot results
    plt.figure(figsize=(12, 5))
    
    # Plot exact values
    plt.plot(results_df['query_index'], results_df['exact_sum'], label='Exact', linewidth=2, color='black')
    
    if not include_initialization:
        plt.xlim((0, len(approx_df) - 1))
    # Plot confidence intervals
    plt.fill_between(results_df['query_index'], 
                     results_df['ci_lb'],
                     results_df['ci_ub'],
                     alpha=0.2,
                     color='red',
                     label='Confidence Interval')
    
    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel('Query Result Sum', fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tight_layout()
    plt.show()
    
    # Calculate summary statistics
    summary = {
        'total_queries': len(results),
        'queries_within_ci': results_df['within_ci'].sum(),
        'avg_actual_error': results_df['actual_error'].mean(),
        'max_actual_error': results_df['actual_error'].max(),
        'avg_error_bound': results_df['error_bound'].mean()
    }
    
    return results_df, summary


def compare_all_error_bounds(directory, include_initialization=False):
    # Load all data using existing function
    df = load_experimental_data(directory)
    df = df[df['run'] == 1]
     # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        conf_lb=('Confidence Interval LB', 'mean'),
        conf_ub=('Confidence Interval UB', 'mean'),
        error_bound=('Error Bound', 'mean'),
        val=('Query Result Sum', 'mean'),
    ).reset_index()
    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]
    
    # Get unique error bounds and sort them
    error_bounds = sorted(agg_df['errorBound'].unique())
    
    # Get exact results (error bound 0)
    exact_df = agg_df[agg_df['errorBound'] == 0]
    
    plt.figure(figsize=(12, 5))
    
    # Plot exact values
    plt.plot(exact_df['i'], exact_df['val'], 
             label='Exact', linewidth=2, color='black')
    
    if not include_initialization:
        plt.xlim((1, 99))
    
    # Plot each error bound result
    # colors = ['red', 'green', 'blue', 'orange', 'purple']  # Add more colors if needed
    for i, error_bound in enumerate([eb for eb in error_bounds if eb != 0]):
        approx_df = agg_df[agg_df['errorBound'] == error_bound]
        
        # Plot confidence intervals
        plt.fill_between(approx_df['i'], 
                        approx_df['conf_lb'],
                        approx_df['conf_ub'],
                        alpha=0.2,
                        color=colors[(i+1) % len(colors)],
                        label=f'{int(error_bound * 100)} %')
        
        
    
    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel('Query Result Sum', fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tight_layout()
    plt.show()

    # Print summary statistics for each error bound
    for error_bound in error_bounds:
        bound_df = agg_df[agg_df['errorBound'] == error_bound]
        if error_bound == 0:
            continue
            
        exact_values = exact_df['val'].values
        estimated_values = (bound_df['conf_lb'] + bound_df['conf_ub'])/2
        actual_errors = np.abs(estimated_values - exact_values)/exact_values
        
        within_bounds = ((bound_df['conf_lb'] <= exact_values) & 
                        (bound_df['conf_ub'] >= exact_values)).mean()
        
        print(f"\nError Bound {error_bound}:")
        print(f"Average Actual Error: {actual_errors.mean():.4f}")
        print(f"Max Actual Error: {actual_errors.max():.4f}")
        print(f"Queries within bounds: {within_bounds*100:.1f}%")


In [ ]:
compare_exact_vs_approx_results(directory, approx_error_bound=0.05)

In [ ]:
compare_all_error_bounds(directory)


# Competitors

In [ ]:
valinor_dir = "synth50/different_errors"
valinor_s_dir = "synth50_only_sampling/different_errors"
default_valinor_a_eb = "0.1"

# Define competitors with their configuration
# Format: {
#   'competitor_name': {
#       'directory': 'path/to/directory',
#       'has_error_bound': True/False,
#       'method_name': 'display_name',
#       'color': 'color_code'
#   }
# }
competitors = {
    'valinor_a': {
        'directory': valinor_dir,
        'has_error_bound': True,
        'method_name': 'Valinor-A',
        'color': '#2ca02c'
    },
    'valinor_s': {
        'directory': valinor_s_dir,
        'has_error_bound': True,
        'method_name': 'Valinor-S',
        'color': '#0000FF'
    },
    'valinor': {
        'directory': valinor_dir,
        'has_error_bound': False,  # Exact method, no error bound
        'method_name': 'Valinor',
        'color': '#d62728'
    },
    'duckdb': {
        'directory': 'taxi_duckdb',  # Update with actual path
        'has_error_bound': False,
        'method_name': 'DuckDB',
        'color': '#ff7f0e'
    },
    'pilotdb': {
        'directory': 'taxi_pilotdb',  # Update with actual path
        'has_error_bound': False,
        'method_name': 'PilotDB',
        'color': "#0efffb"
    }
}

selected_competitors = {
    'valinor_a': competitors['valinor_a'],
    'valinor_s': competitors['valinor_s'],
    'valinor': competitors['valinor']
}



In [ ]:
def get_competitor_agg_df(competitors_config, y_col, eb=None, include_initialization=False):
    """
    Load and aggregate data for multiple competitors.
    
    Args:
        competitors_config: dict of competitors to include {competitor_key: competitor_dict}
        y_col: column name to aggregate
        eb: error bound (string representation, e.g., "0.1")
        include_initialization: whether to include initialization data
    
    Returns:
        DataFrame with aggregated data for all competitors
    """
    all_data = []
    
    for comp_key, comp_config in competitors_config.items():
        directory = comp_config['directory']
        has_error_bound = comp_config['has_error_bound']
        method_name = comp_config['method_name']
        
        try:
            # Load data based on error bound configuration
            if has_error_bound and eb:
                df = load_specific_experimental_data(directory, eb)
            else:
                df = load_specific_experimental_data(directory, "_0_")
            # Aggregate by query
            agg_df = df.groupby(['errorBound', 'i']).agg(
                mean_time=(y_col, 'mean'),
                std_time=(y_col, 'std')
            ).reset_index()
            agg_df['method'] = method_name
            all_data.append(agg_df)
        except Exception as e:
            print(f"Warning: Could not load data for {method_name}: {e}")
            continue
    
    if not all_data:
        raise ValueError("No data could be loaded for the specified competitors")
    
    result_df = pd.concat(all_data, ignore_index=True)
    
    # Optionally exclude initialization
    if not include_initialization:
        result_df = result_df[result_df['i'] != 0]
    
    return result_df


def get_competitor_all_agg_df(competitors_config, y_col, include_initialization=False):
    """
    Load and aggregate data for all error bounds across multiple competitors.
    
    Args:
        competitors_config: dict of competitors to include {competitor_key: competitor_dict}
        y_col: column name to aggregate
        include_initialization: whether to include initialization data
    
    Returns:
        DataFrame with aggregated data for all competitors and error bounds
    """
    all_data = []
    
    for comp_key, comp_config in competitors_config.items():
        directory = comp_config['directory']
        has_error_bound = comp_config['has_error_bound']
        method_name = comp_config['method_name']
        
        try:
            if has_error_bound:
                # Load all error bounds
                df = load_experimental_data(directory)
            else:
                # For methods without error bounds, load exact data once per error bound
                df = load_specific_experimental_data(directory, "_0_")
                # Duplicate for each selected error bound
                temp_data = []
                for eb in selected_error_bounds:
                    df_copy = df.copy()
                    df_copy['errorBound'] = eb
                    temp_data.append(df_copy)
                df = pd.concat(temp_data, ignore_index=True)
            
            # Aggregate by error bound and query
            agg_df = df.groupby(['errorBound', 'i']).agg(
                mean_time=(y_col, 'mean'),
                std_time=(y_col, 'std')
            ).reset_index()
            agg_df['method'] = method_name
            all_data.append(agg_df)
        except Exception as e:
            print(f"Warning: Could not load data for {method_name}: {e}")
            continue
    
    if not all_data:
        raise ValueError("No data could be loaded for the specified competitors")
    
    result_df = pd.concat(all_data, ignore_index=True)
    
    # Optionally exclude initialization
    if not include_initialization:
        result_df = result_df[result_df['i'] != 0]
    
    return result_df


In [ ]:

# Function to plot the aggregated results
def plot_aggregated_competitors_per_column(competitors_config, y_col, eb=None, include_initialization=False):
    """
    Plot competitors as lines across query sequence.
    
    Args:
        competitors_config: dict of competitors to plot
        y_col: column to plot
        eb: error bound (string)
        include_initialization: whether to include initialization
    """
    agg_df = get_competitor_agg_df(competitors_config, y_col, eb, include_initialization)
    
    # Build color palette from competitor config
    color_palette = {}
    for comp_key, comp_config in competitors_config.items():
        method_name = comp_config['method_name']
        color_palette[method_name] = comp_config['color']

    plt.figure(figsize=(12, 5))
    
    # Plot a line for each method
    for method in agg_df['method'].unique():
        data = agg_df[agg_df['method'] == method]
        plt.plot(data['i'], data['mean_time'], 
                 label=method, 
                 linewidth=2,
                 color=color_palette.get(method, '#000000'))

    if not include_initialization:
        plt.xlim((1, 99))
    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel(y_col, fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.xticks(fontsize=fontsize, rotation=0)
    plt.yticks(fontsize=fontsize)

    if not include_initialization:
        plt.xlim(left=1)

    plt.tight_layout()
    plt.show()


def plot_aggregated_competitors_per_column_with_bars(competitors_config, y_col, eb=None, include_initialization=False):
    """
    Plot competitors as lines with a bar chart showing total time.
    
    Args:
        competitors_config: dict of competitors to plot
        y_col: column to plot
        eb: error bound (string)
        include_initialization: whether to include initialization
    """
    agg_df = get_competitor_agg_df(competitors_config, y_col, eb, include_initialization)
    
    # Build color palette
    color_palette = {}
    for comp_key, comp_config in competitors_config.items():
        method_name = comp_config['method_name']
        color_palette[method_name] = comp_config['color']

    fig = plt.figure(figsize=(15, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[3, 1])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # Plot lines on first subplot
    for method in agg_df['method'].unique():
        data = agg_df[agg_df['method'] == method]
        ax1.plot(data['i'], data['mean_time'], 
                 label=method, 
                 linewidth=2,
                 color=color_palette.get(method, '#000000'))

    ax1.set_xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    ax1.set_ylabel(y_col, fontsize=fontsize, fontweight='bold')
    ax1.legend(fontsize=fontsize, frameon=False)
    ax1.grid(False)
    ax1.tick_params(axis='x', labelsize=fontsize, rotation=0)
    ax1.tick_params(axis='y', labelsize=fontsize)

    if not include_initialization:
        ax1.set_xlim((1, 99))

    # Create bar chart on second subplot
    total_times = agg_df.groupby('method')['mean_time'].sum().reset_index()
    total_times = total_times.sort_values(by='mean_time')
    bars = ax2.bar(total_times['method'], total_times['mean_time'], alpha=0.7)

    # Set bar colors
    for bar, method in zip(bars, total_times['method']):
        bar.set_color(color_palette.get(method, '#000000'))

    ax2.set_xlabel('Method', fontsize=fontsize, fontweight='bold')
    ax2.set_ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')
    ax2.tick_params(axis='x', labelsize=fontsize, rotation=0)
    ax2.tick_params(axis='y', labelsize=fontsize)

    plt.tight_layout()
    plt.show()


def plot_aggregated_competitors_per_column_bars_only(competitors_config, y_col, eb=None, include_initialization=False):
    """
    Plot only bar chart showing total time for competitors.
    
    Args:
        competitors_config: dict of competitors to plot
        y_col: column to plot
        eb: error bound (string)
        include_initialization: whether to include initialization
    """
    agg_df = get_competitor_agg_df(competitors_config, y_col, eb, include_initialization)
    
    # Build color palette
    color_palette = {}
    for comp_key, comp_config in competitors_config.items():
        method_name = comp_config['method_name']
        color_palette[method_name] = comp_config['color']

    plt.figure(figsize=(12, 5))
    
    # Create bar chart
    total_times = agg_df.groupby('method')['mean_time'].sum().reset_index()
    total_times = total_times.sort_values(by='mean_time')
    bars = plt.bar(total_times['method'], total_times['mean_time'], alpha=0.7)

    # Set bar colors
    for bar, method in zip(bars, total_times['method']):
        bar.set_color(color_palette.get(method, '#000000'))

    plt.xlabel('Method', fontsize=fontsize, fontweight='bold')
    plt.ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')
    plt.xticks(fontsize=fontsize, rotation=0)
    plt.yticks(fontsize=fontsize)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_aggregated_competitors_per_column(selected_competitors, 'Time (sec)', eb=default_valinor_a_eb)
plot_aggregated_competitors_per_column_bars_only(selected_competitors, 'Time (sec)', eb=default_valinor_a_eb)
plot_aggregated_competitors_per_column_with_bars(selected_competitors, 'Time (sec)', eb=default_valinor_a_eb)

In [ ]:
plot_aggregated_competitors_per_column(selected_competitors, 'I/Os', eb=default_valinor_a_eb)
plot_aggregated_competitors_per_column_bars_only(selected_competitors, 'I/Os', eb=default_valinor_a_eb)
plot_aggregated_competitors_per_column_with_bars(selected_competitors, 'I/Os', eb=default_valinor_a_eb)

In [ ]:
def get_competitor_agg_df_2(competitors_config, eb, include_initialization=False):
    """
    Load and aggregate data for competitors (confidence interval version).
    
    Args:
        competitors_config: dict of competitors to include
        eb: error bound (string representation)
        include_initialization: whether to include initialization
    
    Returns:
        DataFrame with aggregated data
    """
    all_data = []
    
    for comp_key, comp_config in competitors_config.items():
        directory = comp_config['directory']
        has_error_bound = comp_config['has_error_bound']
        method_name = comp_config['method_name']
        
        try:
            # Load data
            if has_error_bound and eb:
                df = load_specific_experimental_data(directory, eb)
            else:
                df = load_specific_experimental_data(directory, "_0_")
            
            df = df[df['run'] == 1]
            
            # Aggregate by query
            
            agg_df = df
            # For methods without error bounds, also get the actual values
            if not has_error_bound:
                agg_df['val'] = df.groupby('i')['Query Result Sum'].mean().values
            else:
                agg_df = df.groupby('i').agg(
                conf_lb=('Confidence Interval LB', 'mean'),
                conf_ub=('Confidence Interval UB', 'mean'),
                error_bound=('Error Bound', 'mean'),
                ).reset_index()
            
            agg_df['method'] = method_name
            all_data.append(agg_df)
        except Exception as e:
            print(f"Warning: Could not load data for {method_name}: {e}")
            continue
    
    if not all_data:
        raise ValueError("No data could be loaded for the specified competitors")
    
    result_df = pd.concat(all_data, ignore_index=True)
    
    # Exclude initialization if needed
    if not include_initialization:
        result_df = result_df[result_df['i'] != 0]
    
    return result_df


In [ ]:
def compare_all_competitor_error_bounds(competitors_config, eb, include_initialization=False):
    """
    Compare all competitors with confidence intervals.
    
    Args:
        competitors_config: dict of competitors to compare
        eb: error bound (string representation, e.g., "0.1")
        include_initialization: whether to include initialization
    """
    # Load data
    agg_df = get_competitor_agg_df_2(competitors_config, eb, include_initialization)
    # Build color palette
    color_palette = {}
    for comp_key, comp_config in competitors_config.items():
        method_name = comp_config['method_name']
        color_palette[method_name] = comp_config['color']
    
    # Get unique methods
    methods = sorted(agg_df['method'].unique())
    
    # Find the exact method (one without error bounds)
    exact_method = None
    for comp_key, comp_config in competitors_config.items():
        if not comp_config['has_error_bound']:
            exact_method = comp_config['method_name']
            break
    
    if exact_method is None:
        raise ValueError("No exact method found in competitors (method with has_error_bound=False)")
    
    exact_df = agg_df[agg_df['method'] == exact_method]

    plt.figure(figsize=(12, 5))
    
    # Plot exact values
    plt.plot(exact_df['i'], exact_df['val'], 
             label='Exact', linewidth=1, color='black')
    
    if not include_initialization:
        plt.xlim((1, 99))
    
    # Plot confidence intervals for other methods
    for method in [m for m in methods if m != exact_method]:
        approx_df = agg_df[agg_df['method'] == method]
        plt.fill_between(approx_df['i'], 
                        approx_df['conf_lb'],
                        approx_df['conf_ub'],
                        alpha=0.2,
                        color=color_palette.get(method, '#000000'),
                        label=method)

    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel('Query Result Sum', fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tight_layout()
    plt.show()

    # Print summary statistics
    for method in methods:
        bound_df = agg_df[agg_df['method'] == method]
        if method == exact_method:
            continue
            
        exact_values = exact_df['val'].values
        estimated_values = (bound_df['conf_lb'] + bound_df['conf_ub'])/2
        actual_errors = np.abs(estimated_values - exact_values)/exact_values
        
        within_bounds = ((bound_df['conf_lb'] <= exact_values) & 
                        (bound_df['conf_ub'] >= exact_values)).mean()
        
        print(f"\n{method}:")
        print(f"Average Actual Error: {actual_errors.mean():.4f}")
        print(f"Max Actual Error: {actual_errors.max():.4f}")
        print(f"Queries within bounds: {within_bounds*100:.1f}%")


In [ ]:
compare_all_competitor_error_bounds(selected_competitors, default_valinor_a_eb)

In [ ]:
def plot_aggregated_competitors_per_column_lines(competitors_config, y_col, include_initialization=False):
    """
    Plot competitors across different error bounds as lines.
    
    Args:
        competitors_config: dict of competitors to plot
        y_col: column to plot
        include_initialization: whether to include initialization
    """
    agg_df = get_competitor_all_agg_df(competitors_config, y_col, include_initialization)
    plt.figure(figsize=(12, 5))

    # Build color palette
    color_palette = {}
    for comp_key, comp_config in competitors_config.items():
        method_name = comp_config['method_name']
        color_palette[method_name] = comp_config['color']
    
    # Calculate total times for each method and error bound
    total_times = agg_df.groupby(['method', 'errorBound'])['mean_time'].sum().reset_index()
    
    # Convert error bounds to percentages and exclude 0
    total_times = total_times[total_times['errorBound'] != 0]
    total_times['errorBound'] = (total_times['errorBound'] * 100).astype(int)
    
    # Sort by error bound
    total_times = total_times.sort_values(by='errorBound', ascending=True)
    
    # Plot lines for each method
    for method in total_times['method'].unique():
        method_data = total_times[total_times['method'] == method]
        plt.plot(method_data['errorBound'].astype(str) + '%', method_data['mean_time'], 
                marker='o', label=method, linewidth=2, markersize=8, 
                color=color_palette.get(method, '#000000'), alpha=0.7)
    
    plt.xlabel('Error Bound (%)', fontsize=fontsize, fontweight='bold')
    plt.ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')
    plt.xticks(fontsize=fontsize, rotation=0)
    plt.yticks(fontsize=fontsize)
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.tight_layout()
    plt.show()


def plot_aggregated_competitors_per_column_bars(competitors_config, y_col, include_initialization=False):
    """
    Plot competitors across different error bounds as grouped bars.
    
    Args:
        competitors_config: dict of competitors to plot
        y_col: column to plot
        include_initialization: whether to include initialization
    """
    agg_df = get_competitor_all_agg_df(competitors_config, y_col, include_initialization)
    plt.figure(figsize=(14, 6))

    # Build color palette
    color_palette = {}
    for comp_key, comp_config in competitors_config.items():
        method_name = comp_config['method_name']
        color_palette[method_name] = comp_config['color']
    
    # Calculate total times for each method and error bound
    total_times = agg_df.groupby(['method', 'errorBound'])['mean_time'].sum().reset_index()
    
    # Convert error bounds to percentages and exclude 0
    total_times = total_times[total_times['errorBound'] != 0]
    total_times['errorBound'] = (total_times['errorBound'] * 100).astype(int)
    
    # Sort by error bound
    total_times = total_times.sort_values(by='errorBound', ascending=True)
    
    # Get unique error bounds and methods
    error_bounds = sorted(total_times['errorBound'].unique(), reverse=True)
    methods = sorted(total_times['method'].unique())
    
    # Set width of bars and positions
    bar_width = 0.25
    x_pos = np.arange(len(error_bounds))

    # Create grouped bars for each method
    for i, method in enumerate(methods):
        method_data = total_times[total_times['method'] == method]
        # Create mapping of error bound to time
        eb_to_time = dict(zip(method_data['errorBound'], method_data['mean_time']))
        
        # Get times for all error bounds
        times = [eb_to_time.get(eb, 0) for eb in error_bounds]
        
        # Plot bars
        positions = x_pos + (i - (len(methods) - 1) / 2) * bar_width
        plt.bar(positions, times, width=bar_width, 
                label=method, color=color_palette.get(method, '#000000'), alpha=0.7)
    
    # Set x-axis ticks
    plt.xticks(x_pos, [f'{eb}%' for eb in error_bounds], fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    
    plt.xlabel('Error Bound (%)', fontsize=fontsize, fontweight='bold')
    plt.ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_aggregated_competitors_per_column_lines(selected_competitors, 'Time (sec)')

In [ ]:
plot_aggregated_competitors_per_column_bars(selected_competitors, 'Time (sec)')

In [ ]:
# Example usage of refactored functions
# ========================================

# Example 1: Select specific competitors (all three)
selected_competitors = {
    'valinor_a': competitors['valinor_a'],
    'valinor_s': competitors['valinor_s'],
    'valinor': competitors['valinor']
}

# Example 2: Include DuckDB competitor
all = {
    'valinor_a': competitors['valinor_a'],
    'valinor_s': competitors['valinor_s'],
    'duckdb': competitors['duckdb'],
    'pilotdb': competitors['pilotdb']
}

# Example 3: Only DuckDB
only_duckdb = {
    'duckdb': competitors['duckdb']
}

# Plot using selected competitors
# plot_aggregated_competitors_per_column(selected_competitors, 'Time (sec)', eb=default_valinor_a_eb)
# plot_aggregated_competitors_per_column_bars_only(selected_competitors, 'Time (sec)', eb=default_valinor_a_eb)
# plot_aggregated_competitors_per_column_with_bars(selected_competitors, 'Time (sec)', eb=default_valinor_a_eb)

# Plot all error bounds across competitors
# plot_aggregated_competitors_per_column_lines(selected_competitors, 'Time (sec)')
# plot_aggregated_competitors_per_column_bars(selected_competitors, 'Time (sec)')

# Compare with confidence intervals
# compare_all_competitor_error_bounds(selected_competitors, default_valinor_a_eb)


In [ ]:
plot_aggregated_competitors_per_column(all, 'Time (sec)', eb=default_valinor_a_eb)
